# Stage 5 — LLM speaker-label correction

**Attach:** `sarvam-diar-code`, `sarvam-diar-attrib` (Stage 4b output),
`sarvam-diar-stage3`.
**Settings:** GPU (T4) on, Internet on.

Stage 4 measured `attribution_cost` (cpWER − DI-cpWER) at 0.01–0.33 across all
twelve conditions. There is no cpWER headroom, so **this stage targets WDER**,
where the spread is real: 20.33 for pyannote31 against 39.53 for
sortformer_stream on identical IndicConformer words.

Three variants are scored side by side — **baseline**, **rule**, **llm** — so
"the LLM helped" has to beat a trivial heuristic, not merely beat doing nothing.

### The edit space, and the invariant it buys

The model may only say *"unit 7 belongs to Speaker_A"*. No new speakers, no moved
boundaries, no edited text.

cpWER, DI-cpWER and WDER are functions of the word→speaker map alone, so relabel
is a **complete** edit space for every metric here — while WER, which ignores
speakers, **cannot move**. A Stage 5 run that shifts WER by 0.01 is broken, and
`check_text_unchanged` asserts it per clip rather than leaving it to be noticed
in the results table.

Units are same-speaker runs split further at pauses > 0.5 s. That split is what
makes false merges reachable at all: two speakers inside one unit cannot be
separated by relabelling it. The audit below measures how much that actually
buys instead of assuming it.

In [23]:
# rapidfuzz is for stage4_score.py's WDER alignment at the end
# of this notebook. It is not preinstalled on Kaggle, and
# discovering that after the GPU work is done wastes a session.
!pip install -q bitsandbytes accelerate rapidfuzz

In [24]:
import pathlib, shutil

ROOT = pathlib.Path("/kaggle/input")
_cands = {p.parent for p in ROOT.rglob("stage5_correct.py")}
if not _cands:
    raise SystemExit("stage5_correct.py is in no attached dataset -- "
                     "re-upload sarvam-diar-code")
CODE = max(_cands, key=lambda d: (len(list(d.glob("stage*.py"))),
                                  "code" in str(d).lower()))
if len(_cands) > 1:
    print("script copies found in:")
    for _c in sorted(map(str, _cands)):
        print("   ", _c, "  <-- using" if str(CODE) == _c else "")

WORK = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)
for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# `attrib` is the one that matters here and it is absent from the Stage 3 and
# Stage 4a datasets, so it joins the list rather than replacing it: the audit
# needs ref/rttm and scoring needs ref/segments.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists()
                            for d in ("attrib", "hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)
if (CODE / "ref").is_dir():
    shutil.copytree(CODE / "ref", WORK / "ref", dirs_exist_ok=True)
    print("restored", CODE / "ref")

print()
print("CODE:", CODE)
CONDS = sorted(p.name for p in (WORK / "attrib").glob("*") if p.is_dir())
print("conditions:", len(CONDS))
for c in CONDS:
    print("   ", c, len(list((WORK / "attrib" / c).glob("*.json"))), "clips")
print("ref/rttm    :", len(list((WORK / "ref" / "rttm").glob("*.rttm"))))
print("ref/segments:", len(list((WORK / "ref" / "segments").glob("*.json"))))

_src = pathlib.Path("/kaggle/working/stage5_correct.py").read_text(encoding="utf-8")
assert "merged_words_maximal" in _src and "def close" in _src, (
    "stale stage5_correct.py -- this copy predates the audit and cleanup fixes. "
    "Re-upload sarvam-diar-code, confirm the new version is attached, and "
    "RESTART THE KERNEL (an already-imported module is not re-read)."
)
print("stage5_correct.py: current")

assert CONDS, "no Stage 4b conditions -- attach the sarvam-diar-attrib dataset"
assert (WORK / "ref" / "rttm").is_dir(), "no ref/rttm: --audit cannot run"
assert (WORK / "ref" / "segments").is_dir(), "no ref/segments: scoring cannot run"

# --- provenance -----------------------------------------------------------
# Kaggle attaches a SPECIFIC dataset version, so an old sarvam-diar-attrib can
# look perfectly healthy while carrying pre-fix words. It cost a full Stage 5
# run once: the directory named `indicconformer` held the unmasked decode, and
# every absolute number came out against the wrong ASR. The comparison between
# baseline/rule/llm was still internally valid, which is exactly what made it
# hard to notice.
#
# `indicconformer_free` was added as an ablation in the same change that fixed
# the multisoftmax decode, so its ABSENCE dates the dataset. That is a
# structural check, not a magic number.
ASRS = sorted({c.split("__")[0] for c in CONDS})
print()
print("ASR systems in attrib:", ASRS)
import json as _json
for c in CONDS:
    _n = sum(len(_json.loads(p.read_text(encoding="utf-8"))["words"])
             for p in sorted((WORK / "attrib" / c).glob("*.json")))
    print(f"   {c:38s} {_n:>7,} words")

if "indicconformer" in ASRS and "indicconformer_free" not in ASRS:
    raise SystemExit(
        "STALE sarvam-diar-attrib: `indicconformer` is present but the "
        "`indicconformer_free` ablation is not. They were produced by the same "
        "change, so this snapshot predates the multisoftmax fix and the words "
        "under `indicconformer` are the UNMASKED decode (WER ~93.7, not "
        "~78.9). Re-run stage4_attribute_score.ipynb, save its output as a new "
        "sarvam-diar-attrib version, and attach THAT version here."
    )

# Stage 5 output must not be corrected again, and the oracle must not be
# corrected at all. These are the conditions worth spending GPU on.
BASE = [c for c in CONDS if "+" not in c and not c.endswith("__ref")]
print()
print("correctable:", BASE)

script copies found in:
    /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code   <-- using
    /kaggle/input/datasets/ritankarmondal/sarvam-diar-stage5 
restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-attrib-score/data
restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-stage5/data
restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/ref

CODE: /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
conditions: 22
    indicconformer__pyannote31 99 clips
    indicconformer__pyannote31+llm 99 clips
    indicconformer__pyannote31+rule 99 clips
    indicconformer__ref 99 clips
    indicconformer__sortformer 74 clips
    indicconformer__sortformer+rule 74 clips
    indicconformer__sortformer_stream 99 clips
    indicconformer__sortformer_stream+rule 99 clips
    indicconformer_free__pyannote31 99 clips
    indicconformer_free__pyannote31+rule 99 clips
    indicconformer_free__ref 99 clips
    indicconformer_free__sortformer

In [25]:
!ls /kaggle/working/data/hyp

pyannote31  sortformer	sortformer_stream


## First: the ceiling

`--audit` reads the reference RTTM and reports how many units still span two
true speakers after pause splitting — the false merges **no relabel can ever
fix** — against how many splitting rescued.

This is an **oracle diagnostic**. It writes no condition, never influences an
edit, and exists so the writeup can state the ceiling on relabel-only correction
rather than imply there isn't one. CPU, seconds.

In [ ]:
!python stage5_correct.py --cond indicconformer__pyannote31 indicconformer__sortformer_stream --audit --data data

## The rule baseline — no GPU

Relabels any unit shorter than 1 s to the neighbour it is closer to, catching the
commonest diarization artefact: a sliver of turn dropped inside someone else's
speech. Run it on every correctable condition; it costs seconds and it is the
number the LLM has to beat.

In [ ]:
# BASE comes from the setup cell: every condition that is not an
# oracle and not already a Stage 5 output.
RULE_ARG = " ".join(BASE)
print("correcting:", RULE_ARG)

In [ ]:
!python stage5_correct.py --method rule --data data --cond {RULE_ARG}

## Look before you spend

Prints the exact prompt for one clip, the model's raw reply, and what survived
parsing and the confidence gate. Two minutes, and it is the cheapest way to find
out that the model returns prose instead of JSON, or wants to relabel everything.

In [ ]:
# # What the model actually sees, and what it says back.
# #
# # Worth two minutes before spending hours: if the reply is not JSON, or the
# # model is relabelling everything, the full run will produce a table of
# # baselines and you will have paid GPU to learn it.

# import importlib, json, pathlib, sys
# sys.path.insert(0, "/kaggle/working")
# import stage5_correct as S

# # A module already imported in this kernel is NOT re-read when the dataset is
# # re-uploaded and the setup cell copies a new file over it -- `import` returns
# # the cached object and you debug a version that is no longer on disk.
# S = importlib.reload(S)

# COND = "indicconformer__pyannote31"
# src = sorted((pathlib.Path("/kaggle/working/data/attrib") / COND).glob("*.json"))[0]
# rec = json.loads(src.read_text(encoding="utf-8"))
# units = S.build_units(rec["words"])
# speakers = sorted({u["spk"] for u in units})
# hi = min(S.WINDOW, len(units))
# prompt = S.build_prompt(units, 0, hi, 0, speakers, rec.get("lang"))

# print(f"clip {rec['clip_id']}  lang={rec.get('lang')}  "
#       f"{len(rec['words'])} words -> {len(units)} units, {len(speakers)} speakers")
# print("=" * 70)
# print(prompt)
# print("=" * 70)

# llm = S.LLM(S.MODEL)
# try:
#     reply = llm(prompt)
#     print("RAW REPLY:")
#     print(reply)
#     print("=" * 70)
#     edits, bad = S.parse_edits(reply, range(0, hi), set(speakers), S.MIN_CONF)
#     print("accepted:", edits)
#     print("rejected:", {k: v for k, v in bad.items() if v})
#     print(f"edit rate: {len(edits)}/{hi} units")
# finally:
#     # MUST run. The weights are ~8.8 GiB and this kernel holds them for its
#     # whole life; the next cell shells out to a SEPARATE python process, which
#     # then has ~5.7 GiB of a 14.6 GiB T4 and dies loading the same model.
#     #
#     # Done inline rather than via llm.close() so this still works against an
#     # older copy of the script -- the cleanup must not itself depend on the
#     # freshness of the thing it is cleaning up after.
#     import gc

#     import torch

#     llm.model = None
#     llm.tok = None
#     del llm
#     gc.collect()
#     torch.cuda.empty_cache()
#     free, total = torch.cuda.mem_get_info()
#     print(f"released GPU memory: {free / 2**30:.1f} of {total / 2**30:.1f} GiB free")

## Smoke test — 3 clips

Watch for: `units` in the tens not hundreds, a **low** edit count (conservative
is correct here), `0 clips rogue`, and no `[WARN]`. A run reporting many
`no confidence given` means the model is ignoring the reply schema — fix the
prompt before reading anything into the score.

In [ ]:
import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")


In [ ]:
!python stage5_correct.py --cond indicconformer__pyannote31 --method llm --data data --limit 99

## Full run

~25 s/clip, so **~40 min per condition**. The four below are ~2.7 h: three ASR
systems on the best diarizer for the results table, plus the worst diarizer on
IndicConformer to test whether correction helps more where there is more to fix.

Resumable — a dead session costs only the clip in flight.

In [ ]:
# !python stage5_correct.py --data data --method llm --cond indicconformer__pyannote31 whisper__pyannote31 indicconformer_free__pyannote31 indicconformer__sortformer_stream

## Two GPUs

Set the accelerator to **T4 x2** and this halves wall-clock. It is data
parallelism, not model sharding: a 4-bit 7B fits in a single T4, and
`device_map="auto"` would split it pipeline-style so the two halves run in
sequence -- slower than one GPU. Instead each worker pins itself to one GPU with
`CUDA_VISIBLE_DEVICES` and takes every other clip.

Shards are computed **before** the resume filter, so each worker owns a fixed
set no matter when it starts, and two workers can never be handed the same clip
-- they would otherwise race on the same output path and double-count in the
manifest.

Output is per-worker log files, printed when both finish; the notebook cell
itself shows nothing until then.

In [ ]:
# # Two GPUs, two clip shards, one condition.
# #
# # NOT model sharding: a 4-bit 7B fits in one T4, and device_map="auto" would
# # split it pipeline-style so the halves run in SEQUENCE -- slower than one GPU,
# # not faster. Data parallelism is where the 2x is: each worker pins itself to
# # one GPU and takes disjoint clips, so neither can touch the other's output.

# import subprocess, sys, pathlib, threading

# COND    = "indicconformer__pyannote31"
# METHOD  = "llm"
# LIMIT   = []          # e.g. ["--limit", "25"] -- applies PER SHARD
# N_GPU   = 2

# def worker(i):
#     cmd = [sys.executable, "stage5_correct.py", "--cond", COND,
#            "--method", METHOD, "--data", "data",
#            "--shard", f"{i}/{N_GPU}"] + LIMIT
#     log = pathlib.Path(f"/kaggle/working/stage5_gpu{i}.log")
#     with log.open("w", encoding="utf-8") as fh:
#         p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT,
#                              cwd="/kaggle/working",
#                              env={**__import__("os").environ,
#                                   "CUDA_VISIBLE_DEVICES": str(i)})
#         rc[i] = p.wait()

# rc = {}
# threads = [threading.Thread(target=worker, args=(i,)) for i in range(N_GPU)]
# for t in threads:
#     t.start()
# print(f"launched {N_GPU} workers; tail the logs below")
# for t in threads:
#     t.join()

# for i in range(N_GPU):
#     print("=" * 70)
#     print(f"GPU {i}  (exit {rc[i]})")
#     print(pathlib.Path(f"/kaggle/working/stage5_gpu{i}.log").read_text(
#         encoding="utf-8", errors="replace")[-2500:])

### What the correction did

In [ ]:
# import json, pathlib, pandas as pd

# rows = []
# for m in sorted(pathlib.Path("/kaggle/working/data/attrib").glob("*+*")):
#     recs = [json.loads(l) for l in
#             (m / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
#     last = {r["clip_id"]: r for r in recs}
#     ok = [r for r in last.values() if r.get("status") == "ok"]
#     if not ok:
#         continue
#     rows.append({
#         "condition": m.name,
#         "clips": len(ok),
#         "units": sum(r["n_units"] for r in ok),
#         "proposed": sum(r["n_proposed"] for r in ok),
#         "applied": sum(r["n_applied"] for r in ok),
#         "words_relabelled": sum(r["n_words_relabelled"] for r in ok),
#         "rogue_clips": sum(int(r["rogue"]) for r in ok),
#         "low_conf": sum(r.get("low_conf", 0) for r in ok),
#         "no_conf": sum(r.get("no_conf", 0) for r in ok),
#         "parse_fail": sum(r.get("parse_fail", 0) for r in ok),
#     })
# df = pd.DataFrame(rows)
# if len(df):
#     df["edit_rate_%"] = (100 * df["applied"] / df["units"]).round(2)
#     df["abstain_%"] = (100 * (df["low_conf"] + df["no_conf"])
#                        / (df["applied"] + df["low_conf"] + df["no_conf"]).clip(lower=1)).round(1)
# print(df.to_string(index=False))

## Score — baseline vs rule vs llm

Stage 4c needs no changes: it splits a condition name on the first `__`, so
`pyannote31+rule` and `pyannote31+llm` read as diarizers and sort directly under
their baseline.

**Check WER first.** A `+rule` or `+llm` row whose WER differs from its baseline
means the text invariant was violated and every other number in the row is void.

In [ ]:
!pip install meeteval

In [ ]:
!python stage4_score.py --data data

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/data/results/asr_summary.csv")
df = df[df.subset == "all"] if "subset" in df else df
print(df.sort_values(["asr", "diar"]).to_string(index=False))

In [26]:
# Stage 5 relabels WORDS, but the results table needs baseline-vs-improved DER and
# JER, which are defined over TIME. This converts the corrected word labels back
# into RTTMs so stage3_score.py can score them.
#
# Boundaries are taken unchanged from the baseline diarizer and only the labels
# are revised, by duration-weighted majority of the corrected words in each
# turn. Rebuilding turns from word spans instead would drop the silence inside
# each turn, shrink hypothesis speech time, and move DER for a reason that has
# nothing to do with Stage 5.

CORRECTED = [c + "+rule" for c in BASE]
RTTM_ARG = " ".join(CORRECTED)
print("converting:", RTTM_ARG)

converting: indicconformer__pyannote31+rule indicconformer__sortformer+rule indicconformer__sortformer_stream+rule indicconformer_free__pyannote31+rule indicconformer_free__sortformer+rule indicconformer_free__sortformer_stream+rule whisper__pyannote31+rule whisper__sortformer+rule whisper__sortformer_stream+rule


In [27]:
!python stage5_to_rttm.py --data data --cond {RTTM_ARG}


[rttm] indicconformer__pyannote31+rule -> hyp/indicconformer__pyannote31+rule
  clips     : 99 written, 0 skipped (no baseline RTTM)
  turns     : 12,809, 2,512 relabelled (19.61%)
  speech    : 11.79 h, 0.88 h relabelled (7.47%)
  boundaries are unchanged, so missed speech, false alarm and total speech time are identical to `pyannote31` -- any DER/JER delta is speaker confusion alone

[rttm] indicconformer__sortformer+rule -> hyp/indicconformer__sortformer+rule
  clips     : 74 written, 0 skipped (no baseline RTTM)
  turns     : 4,793, 849 relabelled (17.71%)
  speech    : 4.63 h, 0.26 h relabelled (5.69%)
  boundaries are unchanged, so missed speech, false alarm and total speech time are identical to `sortformer` -- any DER/JER delta is speaker confusion alone

[rttm] indicconformer__sortformer_stream+rule -> hyp/indicconformer__sortformer_stream+rule
  clips     : 99 written, 0 skipped (no baseline RTTM)
  turns     : 10,426, 2,364 relabelled (22.67%)
  speech    : 12.05 h, 1.00 h 

In [28]:
# Score the relabelled RTTMs against the baselines. Same metric policy as
# Stage 3: collar 0, overlap scored, UEM = the full clip.
import pathlib

SYSTEMS = sorted(p.name for p in pathlib.Path("/kaggle/working/data/hyp").glob("*")
                 if p.is_dir())
print("systems:", SYSTEMS)
DER_ARG = " ".join(SYSTEMS)

systems: ['indicconformer__pyannote31+rule', 'indicconformer__sortformer+rule', 'indicconformer__sortformer_stream+rule', 'indicconformer_free__pyannote31+rule', 'indicconformer_free__sortformer+rule', 'indicconformer_free__sortformer_stream+rule', 'pyannote31', 'sortformer', 'sortformer_stream', 'whisper__pyannote31+rule', 'whisper__sortformer+rule', 'whisper__sortformer_stream+rule']


In [15]:
!pip install -q pyannote.audio pyannote.metrics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 12.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 74.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
!python stage5_to_rttm.py --data data --cond {" ".join(BASE)}


[rttm] indicconformer__pyannote31 -> hyp/indicconformer__pyannote31
  clips     : 99 written, 0 skipped (no baseline RTTM)
  turns     : 12,809, 2,167 relabelled (16.92%)
  speech    : 11.79 h, 0.56 h relabelled (4.79%)
  boundaries are unchanged, so missed speech, false alarm and total speech time are identical to `pyannote31` -- any DER/JER delta is speaker confusion alone

[rttm] indicconformer__sortformer -> hyp/indicconformer__sortformer
  clips     : 74 written, 0 skipped (no baseline RTTM)
  turns     : 4,793, 641 relabelled (13.37%)
  speech    : 4.63 h, 0.17 h relabelled (3.60%)
  boundaries are unchanged, so missed speech, false alarm and total speech time are identical to `sortformer` -- any DER/JER delta is speaker confusion alone

[rttm] indicconformer__sortformer_stream -> hyp/indicconformer__sortformer_stream
  clips     : 99 written, 0 skipped (no baseline RTTM)
  turns     : 10,426, 1,803 relabelled (17.29%)
  speech    : 12.05 h, 0.62 h relabelled (5.13%)
  boundarie

In [33]:
!python stage3_score.py --data data --systems {DER_ARG} --diagnostic

[ok] scored indicconformer__pyannote31: 99 clips
[ok] scored indicconformer__pyannote31+rule: 99 clips
[ok] scored indicconformer__sortformer: 99 clips
[ok] scored indicconformer__sortformer+rule: 99 clips
[ok] scored indicconformer__sortformer_stream: 99 clips
[ok] scored indicconformer__sortformer_stream+rule: 99 clips
[ok] scored indicconformer_free__pyannote31: 99 clips
[ok] scored indicconformer_free__pyannote31+rule: 99 clips
[ok] scored indicconformer_free__sortformer: 99 clips
[ok] scored indicconformer_free__sortformer+rule: 99 clips
[ok] scored indicconformer_free__sortformer_stream: 99 clips
[ok] scored indicconformer_free__sortformer_stream+rule: 99 clips
[ok] scored pyannote31: 99 clips
[ok] scored sortformer: 99 clips
[ok] scored sortformer_stream: 99 clips
[ok] scored whisper__pyannote31: 99 clips
[ok] scored whisper__pyannote31+rule: 99 clips
[ok] scored whisper__sortformer: 99 clips
[ok] scored whisper__sortformer+rule: 99 clips
[ok] scored whisper__sortformer_stream: 

In [34]:
import pandas as pd
print(pd.read_csv("/kaggle/working/data/results/diarization_summary.csv").to_string(index=False))

                                     system  n_clips  n_hyp_missing      DER     miss  false_alarm  confusion  DER_pyannote_accum  JER_pyannote_accum  DER_macro  JER_macro  spk_count_acc  spk_count_mae  spk_count_bias
                 indicconformer__pyannote31       99              0 0.290330 0.115851     0.058910   0.115569            0.290330            0.394688   0.310328   0.391471       0.727273       0.333333       -0.151515
            indicconformer__pyannote31+rule       99              0 0.305525 0.115851     0.058910   0.130764            0.305525            0.410739   0.321687   0.406459       0.737374       0.323232       -0.161616
                 indicconformer__sortformer       99             25 0.752256 0.651379     0.021889   0.078988            0.752256            0.659959   0.528519   0.601341       0.434343       1.555556       -1.414141
            indicconformer__sortformer+rule       99             25 0.754823 0.651379     0.021889   0.081555            0.75482

In [32]:
import pathlib

SYSTEMS = sorted(p.name for p in pathlib.Path("/kaggle/working/data/hyp").glob("*")
                 if p.is_dir())
print("systems:", SYSTEMS)
DER_ARG = " ".join(SYSTEMS)

systems: ['indicconformer__pyannote31', 'indicconformer__pyannote31+rule', 'indicconformer__sortformer', 'indicconformer__sortformer+rule', 'indicconformer__sortformer_stream', 'indicconformer__sortformer_stream+rule', 'indicconformer_free__pyannote31', 'indicconformer_free__pyannote31+rule', 'indicconformer_free__sortformer', 'indicconformer_free__sortformer+rule', 'indicconformer_free__sortformer_stream', 'indicconformer_free__sortformer_stream+rule', 'pyannote31', 'sortformer', 'sortformer_stream', 'whisper__pyannote31', 'whisper__pyannote31+rule', 'whisper__sortformer', 'whisper__sortformer+rule', 'whisper__sortformer_stream', 'whisper__sortformer_stream+rule']


## Save

**Save Version → Quick Save**, then Output tab → **New dataset**, named
`sarvam-diar-stage5`.

In [ ]:
!du -sh /kaggle/working/data/attrib/* 2>/dev/null | tail -20